# 김민준 · RPS 모델 확인 (Google Colab)
사전학습 모델을 준비하고 사진에서 두 손 인식을 확인합니다. 재학습 노트북이 아닙니다.
런타임은 CPU로 충분합니다. 아래 셀을 위에서부터 실행하세요.
실시간 USB 웹캠 게임은 라즈베리파이에서 실행합니다.


In [ ]:
%pip install -q mediapipe==0.10.18 numpy==1.26.4 opencv-contrib-python==4.10.0.84 jax==0.4.35 jaxlib==0.4.35


설치 후 NumPy 관련 오류 또는 재시작 안내가 나오면 런타임을 다시 시작하고 다음 셀부터 실행하세요. Python 버전이 3.13 이상이면 이 고정 버전과 호환되지 않을 수 있으므로 오류를 확인하세요.


In [ ]:
import io, zipfile, urllib.request, pathlib, hashlib
url='https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task'
urllib.request.urlretrieve(url, 'gesture_recognizer.task')
pathlib.Path('models').mkdir(exist_ok=True)
with zipfile.ZipFile('gesture_recognizer.task') as outer:
    for name in outer.namelist():
        with zipfile.ZipFile(io.BytesIO(outer.read(name))) as inner:
            for file in inner.namelist():
                data=inner.read(file)
                assert data[4:8]==b'TFL3', file
                pathlib.Path('models', file).write_bytes(data)
                print(file, len(data), hashlib.sha256(data).hexdigest())


## 사진 한 장으로 확인
양손이 잘 보이는 사진을 선택하세요. 가위는 검지·중지 V 모양입니다. 이 업로드는 Colab에 사진을 전송합니다. 좌우 손 이름은 거울 영상 기준이므로 원본 사진 방향에 따라 다르게 표시될 수 있습니다.


In [ ]:
from google.colab import files
import mediapipe as mp
import cv2
from google.colab.patches import cv2_imshow
uploaded=files.upload()
options=mp.tasks.vision.GestureRecognizerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path='gesture_recognizer.task'), num_hands=2)
with mp.tasks.vision.GestureRecognizer.create_from_options(options) as model:
    for filename in uploaded:
        frame=cv2.imread(filename)
        if frame is None:
            print('사진 파일이 아닙니다:', filename)
            continue
        frame=cv2.flip(frame,1)
        rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        result=model.recognize(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb))
        print(filename, '감지한 손:', len(result.hand_landmarks))
        for hand,gestures in zip(result.handedness,result.gestures):
            print(hand[0].category_name, [(g.category_name,round(g.score,3)) for g in gestures])
        cv2_imshow(frame)


## TFLite 파일 다운로드
프로젝트에는 같은 공식 모델이 이미 포함되어 있습니다. 이 셀은 Colab에서 준비한 파일을 따로 받을 때 사용합니다.


In [ ]:
with zipfile.ZipFile('RPS_TFLite_models.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path('models').glob('*.tflite'):
        z.write(p, str(p))
files.download('RPS_TFLite_models.zip')
